### Configuración del notebook
Variables del Paso 0 (notebook `01_Preparacion`) necesarias para ejecutar este notebook de forma independiente.

In [0]:
CATALOG = "aerolinea_trabajo_final"
RAW_PATH = f"/Volumes/{CATALOG}/landing/raw_files"

### Apartado 1: Ingestión y limpieza básica (capa Bronze)

**Objetivo**: leer los CSV en DataFrames de Spark, convertir tipos, eliminar registros sin claves y guardar en Delta Lake (Bronze).

#### 1.1 Cargar los DataFrames de vuelos, aerolíneas y aeropuertos

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Se leen como texto (sin inferSchema) para controlar la conversión de tipos en el paso 1.2
def leer_csv(nombre):
    return (spark.read
            .option("header", "true")
            .option("inferSchema", "false")
            .csv(f"{RAW_PATH}/{nombre}")
            .withColumn("source_file", F.col("_metadata.file_name")))

flights_raw = leer_csv("flights.csv")
airlines_raw = leer_csv("airlines.csv")
airports_raw = leer_csv("airports.csv")

print("flights :", flights_raw.count(), "filas")
print("airlines:", airlines_raw.count(), "filas")
print("airports:", airports_raw.count(), "filas")
flights_raw.printSchema()
display(flights_raw.limit(10))

flights : 50000 filas
airlines: 6 filas
airports: 10 filas
root
 |-- flight_id: string (nullable = true)
 |-- flight_date: string (nullable = true)
 |-- airline_id: string (nullable = true)
 |-- origin_airport: string (nullable = true)
 |-- destination_airport: string (nullable = true)
 |-- departure_delay: string (nullable = true)
 |-- arrival_delay: string (nullable = true)
 |-- distance_miles: string (nullable = true)
 |-- source_file: string (nullable = false)



flight_id,flight_date,airline_id,origin_airport,destination_airport,departure_delay,arrival_delay,distance_miles,source_file
FL-100000,2025-03-12,NK,LAX,JFK,25.0,29.0,2433,flights.csv
FL-100001,2025-01-09,AA,LAS,MIA,18.0,38.0,308,flights.csv
FL-100002,2025-04-18,B6,ATL,LAS,10.0,13.0,226,flights.csv
FL-100003,2025-03-13,DL,MIA,SFO,37.0,49.0,579,flights.csv
FL-100004,2025-03-30,WN,LAX,MIA,-4.0,6.0,2396,flights.csv
FL-100005,2025-05-22,AA,MIA,LAX,-4.0,-4.0,1681,flights.csv
FL-100006,2025-01-12,B6,ATL,LAX,20.0,6.0,1153,flights.csv
FL-100007,2025-04-27,AA,MIA,DFW,-29.0,-30.0,1058,flights.csv
FL-100008,2025-06-05,NK,DFW,LAX,-5.0,3.0,2093,flights.csv
FL-100009,2025-02-26,WN,DFW,LAS,-13.0,-22.0,1138,flights.csv


#### 1.2 Convertir tipos de columnas

- `flight_date` → `DateType`
- `departure_delay` y `arrival_delay` → `IntegerType` (se renombran a `dep_delay` y `arr_delay`, los nombres que usa el enunciado)
- `distance_miles` → `IntegerType`

In [0]:
flights_typed = (flights_raw
    .withColumn("flight_date", F.to_date("flight_date", "yyyy-MM-dd"))
    # Los retrasos vienen como "25.0": se pasan a double y luego a entero
    .withColumn("dep_delay", F.col("departure_delay").cast("double").cast("int"))
    .withColumn("arr_delay", F.col("arrival_delay").cast("double").cast("int"))
    .withColumn("distance_miles", F.col("distance_miles").cast("int"))
    .withColumn("airline_id", F.trim(F.upper("airline_id")))
    .withColumn("origin_airport", F.trim(F.upper("origin_airport")))
    .withColumn("destination_airport", F.trim(F.upper("destination_airport")))
    .drop("departure_delay", "arrival_delay")
    .select("flight_id", "flight_date", "airline_id", "origin_airport",
            "destination_airport", "dep_delay", "arr_delay", "distance_miles", "source_file"))

airlines_typed = airlines_raw.select(
    F.trim(F.upper("airline_id")).alias("airline_id"),
    F.trim("airline_name").alias("airline_name"),
    "source_file")

airports_typed = airports_raw.select(
    F.trim(F.upper("airport_id")).alias("airport_id"),
    F.trim("city").alias("city"),
    F.trim("country").alias("country"),
    "source_file")

flights_typed.printSchema()

root
 |-- flight_id: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- airline_id: string (nullable = true)
 |-- origin_airport: string (nullable = true)
 |-- destination_airport: string (nullable = true)
 |-- dep_delay: integer (nullable = true)
 |-- arr_delay: integer (nullable = true)
 |-- distance_miles: integer (nullable = true)
 |-- source_file: string (nullable = false)



#### 1.3 Eliminar filas con nulos en `flight_id`, `airline_id` u `origin_airport`

In [0]:
antes = flights_typed.count()

# Conteo de nulos por columna antes de limpiar
display(flights_typed.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in flights_typed.columns]))

flights_clean = (flights_typed
    .na.drop(subset=["flight_id", "airline_id", "origin_airport"])
    .dropDuplicates(["flight_id"]))
airlines_clean = airlines_typed.na.drop(subset=["airline_id"]).dropDuplicates(["airline_id"])
airports_clean = airports_typed.na.drop(subset=["airport_id"]).dropDuplicates(["airport_id"])

despues = flights_clean.count()
print(f"Vuelos antes: {antes} | después: {despues} | eliminados: {antes - despues}")

flight_id,flight_date,airline_id,origin_airport,destination_airport,dep_delay,arr_delay,distance_miles,source_file
521,0,0,0,0,244,236,0,0


Vuelos antes: 50000 | después: 49479 | eliminados: 521


#### 1.4 Agregar la columna `ingest_timestamp`

In [0]:
flights_bronze = flights_clean.withColumn("ingest_timestamp", F.current_timestamp())
airlines_bronze = airlines_clean.withColumn("ingest_timestamp", F.current_timestamp())
airports_bronze = airports_clean.withColumn("ingest_timestamp", F.current_timestamp())
display(flights_bronze.limit(5))

flight_id,flight_date,airline_id,origin_airport,destination_airport,dep_delay,arr_delay,distance_miles,source_file,ingest_timestamp
FL-100022,2025-01-31,WN,SEA,DEN,-25,-27,2468,flights.csv,2026-11-05T00:17:41.498Z
FL-100039,2025-02-10,DL,SFO,DFW,7,-8,2366,flights.csv,2026-11-05T00:17:41.498Z
FL-100068,2025-01-17,NK,ORD,DFW,-1,1,455,flights.csv,2026-11-05T00:17:41.498Z
FL-100072,2025-04-30,NK,MIA,ORD,20,33,1813,flights.csv,2026-11-05T00:17:41.498Z
FL-100073,2025-01-22,B6,LAS,JFK,-10,-18,405,flights.csv,2026-11-05T00:17:41.498Z


#### 1.5 Guardar en Delta Lake (capa Bronze)

In [0]:
def guardar_delta(df, schema, tabla):
    nombre = f"{CATALOG}.{schema}.{tabla}"
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(nombre))
    print(f"Guardada {nombre}: {spark.table(nombre).count()} filas")

guardar_delta(flights_bronze, "bronze", "flights")
guardar_delta(airlines_bronze, "bronze", "airlines")
guardar_delta(airports_bronze, "bronze", "airports")

Guardada aerolinea_trabajo_final.bronze.flights: 49479 filas
Guardada aerolinea_trabajo_final.bronze.airlines: 6 filas
Guardada aerolinea_trabajo_final.bronze.airports: 10 filas
